# Predicted Epitopes of Trypanosoma cruzi Exploration with `mlcroissant`
This notebook provides an interactive guide for loading and exploring the "Predicted Epitopes of Trypanosoma cruzi based on Phage Display Data" dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is described by a Croissant schema available at the provided URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and inspect basic properties from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.etbd-dths/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_json = dataset.metadata.to_json()
print("Dataset Name:", metadata_json['name'])
print("Description:", metadata_json['description'])
print("License:", metadata_json['license'])
print("Version:", metadata_json['version'])

## 2. Data Overview
Review available record sets, their field `@id`s, and a sample of their records.

All entities are referenced by their full `@id` from the Croissant schema.

In [ ]:
# List all available record set @id's

record_sets = dataset.record_sets()
print(f"Record sets found ({len(record_sets)}):")
for idx, rset in enumerate(record_sets):
    print(f"[{idx}] @id = {rset['@id']}")
    print(f"    Name: {rset.get('name', '(no name)')}")
    fields = rset.get('field', [])
    if isinstance(fields, dict):
        # In case only one field
        fields = [fields]
    print(f"    Fields [@id's]:", [f['@id'] for f in fields])
    print()

## 3. Data Extraction
Load data from selected record sets into DataFrames for analysis.

All access is by entity `@id` (record set, field, and column).

In [ ]:
# You may choose one or more record sets to explore. Here we select all.

dataframes = dict()

for rset in record_sets:
    rset_id = rset['@id']
    print(f"Loading data for record set: {rset_id}")
    try:
        records = list(dataset.records(record_set=rset_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rset_id] = df
            print(f"  Loaded {len(df)} records. Columns:", df.columns.tolist())
            print(df.head(2))
        else:
            print("  No records available.")
    except Exception as e:
        print(f"  Could not load records for {rset_id}: {e}")
    print()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records, normalizing numeric fields, and grouping by key attributes.

In this example, we'll pick the first record set and, if it has a numeric field, apply filtering and normalization by referencing the field `@id`.

In [ ]:
from pandas.api.types import is_numeric_dtype

# Select the first record set with data
if len(dataframes) == 0:
    print("No record sets contain tabular data with records.")
else:
    # Choose the first record set with actual data
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Selected record set: {record_set_id}")
    print(df.head())
    # Pick a numeric field if available
    numeric_field = None
    for col in df.columns:
        if is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if not numeric_field:
        print("No numeric field found for EDA.")
    else:
        print(f"Using numeric field for filtering: {numeric_field}")
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (mean): {len(filtered_df)} records")
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Optionally, group by another field (if available and categorical)
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].nunique() < min(20, len(df) // 2):
                group_field = col
                break
        if group_field:
            grouped = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped by {group_field}:\n", grouped.head())

## 5. Visualization
Visualize a distribution (histogram) or relationship if the data allows. We'll plot the selected numeric field if available.

In [ ]:
import matplotlib.pyplot as plt

if len(dataframes) > 0:
    # Use the variables found in the last cell
    df = dataframes[record_set_id]
    if 'numeric_field' in locals() and numeric_field is not None:
        plt.figure(figsize=(7,4))
        df[numeric_field].hist(bins=30)
        plt.title(f"Distribution of {numeric_field} in {record_set_id}")
        plt.xlabel(numeric_field)
        plt.ylabel("Frequency")
        plt.show()

## 6. Conclusion
This notebook demonstrated loading, exploring, and basic processing of the "Predicted Epitopes of Trypanosoma cruzi" dataset using `mlcroissant`.

**Key steps illustrated:**
- Metadata access and dataset description
- Enumerating and selecting record sets (by `@id` per Croissant schema)
- Loading records into dataframes for analysis
- Filtering, normalization, and basic grouping of numeric fields
- Visualizing distributions of values

Further analysis can utilize specific field `@id` references and the rich annotation in the Croissant schema for more advanced studies.